# QuixBugs — Plausibility Testing

Runs the QuixBugs inline `assert` test cases against generated patches and reports **plausible@K** — the gold-standard repair metric.

**Inputs:**
- `data/quixbugs_eval.jsonl` — eval set (with `tests` field)
- `results/quixbugs_<model>_baseline.jsonl` — generations from `01_baseline_codellama.ipynb` (or any inference run with the same schema)

**Output:** `results/quixbugs_<model>_plausibility.jsonl` — one line per (bug_id, gen_idx) with test_pass / test_status.

**Parameters:** `START_BUG`, `END_BUG` — slice the eval set by index. Run in batches if you want; rerun is **resume-safe** (existing rows are skipped).

**Time:** Negligible — pure Python subprocess, ~0.5s per generation × 400 = ~3 minutes for the full set. Doesn't need a GPU.

## 1. Setup — clone repo & install (no GPU needed)

In [1]:
print("Kernel Connected")

Kernel Connected


In [4]:
!cd "d:\snake-repairllama-baseline"

 Volume in drive D is DATA
 Volume Serial Number is DEB3-B710

 Directory of d:\snake-repairllama-baseline\notebooks

05/05/2026  11:44 PM    <DIR>          .
05/08/2026  11:55 PM    <DIR>          ..
05/08/2026  09:53 PM            92,346 01_baseline_codellama.ipynb
04/26/2026  08:12 PM             7,610 02_plausibility_quixbugs.ipynb
04/26/2026  11:08 PM             9,714 03_plausibility_bugsinpy.ipynb
05/08/2026  01:43 AM            99,728 04_run1_snakellama.ipynb
               4 File(s)        209,398 bytes
               2 Dir(s)  56,554,074,112 bytes free


In [2]:
import os, sys, subprocess

REPO_URL = "https://github.com/AliSuleman27/snake-repairllama-baseline.git"
# REPO_DIR = "/content/snake-repairllama-baseline"
REPO_DIR = "d:/snake-repairllama-baseline"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())

cwd: d:\snake-repairllama-baseline


## 2. Provide the generations file

The notebook expects a generations JSONL produced by `src/inference.py` (one line per bug, with a `generations` list of 10 strings).

**Option A** — mount Drive and point to your saved file:

In [ ]:
# Option A: pull from Google Drive (run this if your generations are on Drive)
from google.colab import drive
import shutil, os
drive.mount("/content/drive", force_remount=False)

DRIVE_GENS = "/content/drive/MyDrive/snake-repairllama-baseline-results/quixbugs_codellama_baseline.jsonl"
LOCAL_GENS = "results/quixbugs_codellama_baseline.jsonl"

if os.path.exists(DRIVE_GENS):
    os.makedirs("results", exist_ok=True)
    shutil.copy(DRIVE_GENS, LOCAL_GENS)
    print("copied:", LOCAL_GENS)
else:
    print("WARN: Drive file not found at", DRIVE_GENS)
    print("      Either change the path above, or upload manually with the Files panel.")

In [ ]:
# Option B: upload manually via the Colab Files panel into results/
# (skip this cell if Option A worked)
import os
os.makedirs("results", exist_ok=True)
print("results/ exists. Upload your <model>_baseline.jsonl into it via the file panel.")
print("Existing files in results/:", os.listdir("results"))

## 3. Parameters

Edit these and re-run the next cell. The total set is 40 bugs (indices 0..39).

In [3]:
# BASE_PATH = ""
import os
BASE_PATH = "d:/snake-repairllama-baseline/"
EVAL_FILE        = BASE_PATH + "data/quixbugs_eval.jsonl"
GENERATIONS_FILE = BASE_PATH + "results/quixbugs_snakellama_run3.jsonl"  # change me if your model name differs
OUTPUT_FILE      = BASE_PATH + "results/quixbugs_snakellama_plausibility.jsonl"

START_BUG = 0    # i  (inclusive, 0-based index into eval file)
END_BUG   = 40   # j  (exclusive). Set to None to run to the end.
TIMEOUT   = 10   # seconds per patch test
RESUME    = True

# Sanity check inputs exist
for p in (EVAL_FILE, GENERATIONS_FILE):
    assert os.path.exists(p), f"missing: {p}"
print("All inputs present.")

All inputs present.


## 4. Run plausibility tests on the slice

In [4]:
from src.runners.quixbugs import run_plausibility

run_plausibility(
    eval_jsonl=EVAL_FILE,
    inference_jsonl=GENERATIONS_FILE,
    output_jsonl=OUTPUT_FILE,
    start_bug=START_BUG,
    end_bug=END_BUG,
    timeout_sec=TIMEOUT,
    resume=RESUME,
)

[quixbugs] Targeting bugs 0..40 (40 bugs)
  quixbugs/bitcount                    done
  quixbugs/breadth_first_search        done
  quixbugs/bucketsort                  done
  quixbugs/depth_first_search          done
  quixbugs/detect_cycle                done
  quixbugs/find_first_in_sorted        done
  quixbugs/find_in_sorted              done
  quixbugs/flatten                     done
  quixbugs/gcd                         done
  quixbugs/get_factors                 done
  quixbugs/hanoi                       done
  quixbugs/is_valid_parenthesization   done
  quixbugs/kheapsort                   done
  quixbugs/knapsack                    done
  quixbugs/kth                         done
  quixbugs/lcs_length                  done
  quixbugs/levenshtein                 done
  quixbugs/lis                         done
  quixbugs/longest_common_subsequence  done
  quixbugs/max_sublist_sum             done
  quixbugs/mergesort                   done
  quixbugs/minimum_spanning_tree  

## 5. Aggregate report (across whatever's been tested so far)

This includes plausible@1/3/10 alongside the existing exact/AST/compile/buried metrics.

In [5]:
from src.metrics import evaluate_file, print_report

result = evaluate_file(
    inference_jsonl=GENERATIONS_FILE,
    eval_jsonl=EVAL_FILE,
    plausibility_jsonl=OUTPUT_FILE,
)
print_report("QuixBugs — vanilla CodeLlama-7B-Python (with plausibility)", result)


  QuixBugs — vanilla CodeLlama-7B-Python (with plausibility)
  Total bugs: 40    Bugs with plausibility data: 40
  Top-1  Exact     :   19 / 40 ( 47.5%)
  Top-1  AST       :   21 / 40 ( 52.5%)
  Top-1  Compile   :   25 / 40 ( 62.5%)
  Top-1  Plausible :   23 / 40 ( 57.5%) [tests passed]

  Top-3  Exact     :   28 / 40 ( 70.0%)
  Top-3  AST       :   30 / 40 ( 75.0%)
  Top-3  Compile   :   26 / 40 ( 65.0%)
  Top-3  Buried    :   28 / 40 ( 70.0%)
  Top-3  Plausible :   34 / 40 ( 85.0%)

  Top-10 Exact     :   32 / 40 ( 80.0%)
  Top-10 AST       :   33 / 40 ( 82.5%)
  Top-10 Compile   :   29 / 40 ( 72.5%)
  Top-10 Buried    :   32 / 40 ( 80.0%)
  Top-10 Plausible :   36 / 40 ( 90.0%)


## 6. (Optional) Per-bug breakdown — see exactly which patches passed

In [6]:
import json
from collections import Counter

by_bug = {}
with open(OUTPUT_FILE) as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        by_bug.setdefault(r["bug_id"], []).append(r)

print(f"{'bug_id':40s}  {'pass/total':>10s}  status_counts")
print("-" * 90)
for bug_id in sorted(by_bug):
    rows = by_bug[bug_id]
    n_pass = sum(r['test_pass'] for r in rows)
    statuses = Counter(r['test_status'] for r in rows)
    status_str = " ".join(f"{s}={c}" for s, c in statuses.most_common())
    print(f"{bug_id:40s}  {n_pass:>4}/{len(rows):>4}    {status_str}")

bug_id                                    pass/total  status_counts
------------------------------------------------------------------------------------------
quixbugs/bitcount                            8/  10    pass=8 timeout=1 compile=1
quixbugs/breadth_first_search               10/  10    pass=10
quixbugs/bucketsort                         10/  10    pass=10
quixbugs/depth_first_search                  3/  10    error=6 pass=3 fail=1
quixbugs/detect_cycle                        8/  10    pass=8 error=2
quixbugs/find_first_in_sorted               10/  10    pass=10
quixbugs/find_in_sorted                      8/  10    pass=8 error=2
quixbugs/flatten                            10/  10    pass=10
quixbugs/gcd                                10/  10    pass=10
quixbugs/get_factors                         5/  10    fail=5 pass=5
quixbugs/hanoi                               4/  10    fail=5 pass=4 error=1
quixbugs/is_valid_parenthesization          10/  10    pass=10
quixbugs/kheapsort

## 7. (Optional) Save plausibility JSONL back to Drive

In [ ]:
import shutil, os
dest = "/content/drive/MyDrive/snake-repairllama-baseline-results"
os.makedirs(dest, exist_ok=True)
shutil.copy(OUTPUT_FILE, dest + "/" + os.path.basename(OUTPUT_FILE))
print("saved to Drive:", dest)